In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2

# -------------------------------
# 1️⃣ Dataset for unlabeled images
# -------------------------------
class CancerDataset(Dataset):
    def __init__(self, img_paths, transform=None):
        self.img_paths = img_paths
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path = self.img_paths[idx]
        img = Image.open(path).convert("RGB")  # ResNet expects 3 channels
        if self.transform:
            img = self.transform(img)
        return img, path  # return path to display filename if needed

# -------------------------------
# 2️⃣ Load images
# -------------------------------
data_dir = "/home/latifa/Téléchargements/imgdatabreastcancer/jpeg"
image_paths = [os.path.join(data_dir, f) for f in os.listdir(data_dir)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = CancerDataset(image_paths, transform=transform)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

# -------------------------------
# 3️⃣ Load pretrained ResNet18
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 2)  # still 2-class output
model = model.to(device)
model.eval()

# -------------------------------
# 4️⃣ Grad-CAM function
# -------------------------------
def generate_grad_cam(model, image, target_class, device):
    model.eval()
    gradients = []
    activations = []

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0].cpu().data.numpy())

    def forward_hook(module, input, output):
        activations.append(output.cpu().data.numpy())

    last_conv = model.layer4[1].conv2
    forward_handle = last_conv.register_forward_hook(forward_hook)
    backward_handle = last_conv.register_backward_hook(backward_hook)

    image = image.unsqueeze(0).to(device)
    output = model(image)
    model.zero_grad()
    class_loss = output[0, target_class]
    class_loss.backward()

    grads = gradients[0][0]
    acts = activations[0][0]
    weights = np.mean(grads, axis=(1,2))
    cam = np.zeros(acts.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        cam += w * acts[i]

    cam = np.maximum(cam, 0)
    cam = cv2.resize(cam, (image.size(2), image.size(3)))
    cam = cam - np.min(cam)
    cam = cam / np.max(cam)

    forward_handle.remove()
    backward_handle.remove()
    return cam

# -------------------------------
# 5️⃣ Generate Grad-CAM for each image
# -------------------------------
for img, path in dataloader:
    img = img.to(device)
    cam = generate_grad_cam(model, img[0], target_class=0, device=device)  # target_class can be 0 or 1

    # Convert tensor to numpy for display
    img_np = img[0].cpu().permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    plt.imshow(img_np)
    plt.imshow(cam, cmap='jet', alpha=0.5)
    plt.title(os.path.basename(path[0]))
    plt.axis('off')
    plt.show()
